## Stage 3 preview: baseline and fine-tuning

In [16]:
from google.colab import drive
drive.mount("/content/drive")
!pip -q install -U sentence-transformers datasets accelerate

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [17]:
import json, pandas as pd, torch
D = "/content/drive/MyDrive/sanskrit_retrieval/data"
load = lambda p: [json.loads(l) for l in open(p, encoding="utf-8")]

train_pairs = load(f"{D}/train_pairs.jsonl")
val_eval  = pd.DataFrame(load(f"{D}/val_eval.jsonl"))
test_eval = pd.DataFrame(load(f"{D}/test_eval.jsonl"))
val  = pd.read_csv(f"{D}/gita_val.csv",  dtype={"id": str})
test = pd.read_csv(f"{D}/gita_test.csv", dtype={"id": str})

In [18]:
print(len(train_pairs), len(val_eval), len(test_eval))   # expect 5669, 772, 1144
print("cuda:", torch.cuda.is_available())

x = torch.randn(4, 4, device="cuda"); print((x @ x).sum())   # real GPU test

5669 772 1144
cuda: True
tensor(-3.5912, device='cuda:0')
5669 772 1144
cuda: True
tensor(7.2043, device='cuda:0')


# Step 3.1: baseline on val

In [19]:
import numpy as np, os
from sentence_transformers import SentenceTransformer

RES = "/content/drive/MyDrive/sanskrit_retrieval/results"
os.makedirs(RES, exist_ok=True)

def evaluate(model, ev, corpus, qp="query: ", dp="passage: "):
    ids = corpus.id.tolist(); idx = {i: k for k, i in enumerate(ids)}
    text = {"english": corpus.english.tolist(), "deva": corpus.deva.tolist()}
    enc = lambda xs: model.encode(xs, normalize_embeddings=True, batch_size=64, show_progress_bar=False)
    docs = {s: enc([dp + t for t in text[s]]) for s in text}
    Q = enc([qp + q for q in ev["query"]])
    ranks = []
    for q, side, gold in zip(Q, ev["doc_side"], ev["gold_id"]):
        sims = docs[side] @ q
        ranks.append(int((sims > sims[idx[gold]]).sum()) + 1)
    out = ev.copy(); out["rank"] = ranks
    return out

def summarize(out):
    rows = {}
    for name, g in list(out.groupby("qtype")) + [("ALL", out)]:
        r = g["rank"].to_numpy()
        rows[name] = {"n": len(r), "R@1": (r <= 1).mean(), "R@5": (r <= 5).mean(),
                      "R@10": (r <= 10).mean(), "MRR": (1 / r).mean(),
                      "nDCG@10": np.where(r <= 10, 1 / np.log2(r + 1), 0).mean()}
    return pd.DataFrame(rows).T.round(3)

base = SentenceTransformer("intfloat/multilingual-e5-small", device="cuda")
base_out = evaluate(base, val_eval, val)
base_tab = summarize(base_out)
base_out.to_csv(f"{RES}/baseline_val_ranks.csv", index=False)
base_tab.to_csv(f"{RES}/baseline_val_metrics.csv")
base_tab

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

,n,R@1,R@5,R@10,MRR,nDCG@10
ascii_full,71.0,0.014,0.085,0.225,0.077,0.090
ascii_half,68.0,0.015,0.103,0.191,0.088,0.091
ascii_short,71.0,0.028,0.099,0.211,0.095,0.101
ascii_to_deva,71.0,0.239,0.521,0.732,0.376,0.449
deva_full,71.0,0.113,0.310,0.563,0.233,0.294
deva_half,68.0,0.088,0.382,0.515,0.227,0.280
english_to_deva,71.0,0.296,0.577,0.690,0.424,0.478
iast_full,71.0,0.028,0.099,0.239,0.089,0.103
q_casual,69.0,0.290,0.478,0.638,0.407,0.448
q_keyword,71.0,0.268,0.563,0.662,0.413,0.460


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

,n,R@1,R@5,R@10,MRR,nDCG@10
ascii_full,71.0,0.014,0.085,0.225,0.077,0.090
ascii_half,68.0,0.015,0.103,0.191,0.088,0.091
ascii_short,71.0,0.028,0.099,0.211,0.095,0.101
ascii_to_deva,71.0,0.239,0.521,0.732,0.376,0.449
deva_full,71.0,0.113,0.310,0.563,0.233,0.294
deva_half,68.0,0.088,0.382,0.515,0.227,0.280
english_to_deva,71.0,0.296,0.577,0.690,0.424,0.478
iast_full,71.0,0.028,0.099,0.239,0.089,0.103
q_casual,69.0,0.290,0.478,0.638,0.407,0.448
q_keyword,71.0,0.268,0.563,0.662,0.413,0.460


# Step 3.2: first fine-tuning run

In [20]:
import random
from datasets import Dataset
from sentence_transformers import (SentenceTransformer, SentenceTransformerTrainer,
                                   SentenceTransformerTrainingArguments)
from sentence_transformers.losses import MultipleNegativesRankingLoss
from sentence_transformers.training_args import BatchSamplers

rng = random.Random(42)
rows = [{"anchor":   "query: "   + p["query"],
         "positive": "passage: " + p["positive"],
         "negative": "passage: " + rng.choice(p["negatives"])} for p in train_pairs]
ds = Dataset.from_list(rows).shuffle(seed=42)

model = SentenceTransformer("intfloat/multilingual-e5-small", device="cuda")
model.max_seq_length = 256      # truncates the few very long verses; saves memory

OUT = "/content/drive/MyDrive/sanskrit_retrieval/models/e5_ft_v1"
args = SentenceTransformerTrainingArguments(
    output_dir=OUT, num_train_epochs=3, per_device_train_batch_size=32,
    learning_rate=2e-5, warmup_ratio=0.1, fp16=True,
    batch_sampler=BatchSamplers.NO_DUPLICATES,
    save_strategy="epoch", save_total_limit=3,
    logging_steps=20, report_to="none", seed=42)

trainer = SentenceTransformerTrainer(model=model, args=args, train_dataset=ds,
                                     loss=MultipleNegativesRankingLoss(model))
trainer.train()
model.save_pretrained(f"{OUT}/final")

/tmp/ipykernel_4418/2908128690.py:5: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers.losses import MultipleNegativesRankingLoss
/tmp/ipykernel_4418/2908128690.py:6: DeprecationWarning: Importing from 'sentence_transformers.training_args' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.training_args' instead.
  from sentence_transformers.training_args import BatchSamplers


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 0, 'pad_token_id': 1}.


Step,Training Loss
20,3.852419
40,3.611590
60,3.116555
80,2.640981
100,2.358527
120,2.229683
140,2.085194
160,1.931849
180,2.038922
200,1.854329


...

Step,Training Loss
20,3.852428
40,3.611593
60,3.116542
80,2.641074
100,2.358531
120,2.229657
140,2.085170
160,1.931778
180,2.038909
200,1.854381


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [21]:
ft = SentenceTransformer(f"{OUT}/final", device="cuda")
ft_out = evaluate(ft, val_eval, val)
ft_tab = summarize(ft_out)
ft_out.to_csv(f"{RES}/ft_v1_val_ranks.csv", index=False)
ft_tab.to_csv(f"{RES}/ft_v1_val_metrics.csv")

cmp = pd.concat({"base": base_tab[["R@1", "MRR"]], "ft": ft_tab[["R@1", "MRR"]]}, axis=1)
cmp["ΔMRR"] = cmp[("ft", "MRR")] - cmp[("base", "MRR")]
cmp.round(3)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

base            ft          ΔMRR
                   R@1    MRR    R@1    MRR       
ascii_full       0.014  0.077  0.141  0.247  0.170
ascii_half       0.015  0.088  0.088  0.196  0.108
ascii_short      0.028  0.095  0.127  0.219  0.124
ascii_to_deva    0.239  0.376  0.817  0.862  0.486
deva_full        0.113  0.233  0.465  0.586  0.353
deva_half        0.088  0.227  0.294  0.446  0.219
english_to_deva  0.296  0.424  0.479  0.590  0.166
iast_full        0.028  0.089  0.141  0.253  0.164
q_casual         0.290  0.407  0.333  0.499  0.092
q_keyword        0.268  0.413  0.479  0.603  0.190
q_natural        0.186  0.303  0.400  0.546  0.243
ALL              0.142  0.249  0.343  0.460  0.211

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

base            ft          ΔMRR
                   R@1    MRR    R@1    MRR       
ascii_full       0.014  0.077  0.141  0.247  0.170
ascii_half       0.015  0.088  0.088  0.196  0.108
ascii_short      0.028  0.095  0.127  0.219  0.124
ascii_to_deva    0.239  0.376  0.817  0.862  0.486
deva_full        0.113  0.233  0.465  0.586  0.353
deva_half        0.088  0.227  0.294  0.446  0.219
english_to_deva  0.296  0.424  0.479  0.590  0.166
iast_full        0.028  0.089  0.141  0.253  0.164
q_casual         0.290  0.407  0.333  0.499  0.092
q_keyword        0.268  0.413  0.479  0.603  0.190
q_natural        0.186  0.303  0.400  0.546  0.243
ALL              0.142  0.249  0.343  0.460  0.211

# Step 3.3: two ablations
1. No hard negatives (in-batch negatives only): do hard negatives help?
2. No romanized query types (drops all ASCII/IAST queries): does transliteration training help?

In [22]:
import gc
ROOT = "/content/drive/MyDrive/sanskrit_retrieval"

def make_ds(pairs, use_hard=True, drop=()):
    rng = random.Random(42); rows = []
    for p in pairs:
        if p["qtype"] in drop: continue
        r = {"anchor": "query: " + p["query"], "positive": "passage: " + p["positive"]}
        if use_hard: r["negative"] = "passage: " + rng.choice(p["negatives"])
        rows.append(r)
    return Dataset.from_list(rows).shuffle(seed=42)

def run(name, ds):
    model = SentenceTransformer("intfloat/multilingual-e5-small", device="cuda")
    model.max_seq_length = 256
    args = SentenceTransformerTrainingArguments(
        output_dir=f"{ROOT}/models/{name}", num_train_epochs=3, per_device_train_batch_size=32,
        learning_rate=2e-5, warmup_ratio=0.1, fp16=True, batch_sampler=BatchSamplers.NO_DUPLICATES,
        save_strategy="no", logging_steps=20, report_to="none", seed=42)
    SentenceTransformerTrainer(model=model, args=args, train_dataset=ds,
                               loss=MultipleNegativesRankingLoss(model)).train()
    tab = summarize(evaluate(model, val_eval, val))
    tab.to_csv(f"{RES}/{name}_val_metrics.csv")
    del model; gc.collect(); torch.cuda.empty_cache()
    return tab

ROMAN = ("ascii_full", "ascii_half", "ascii_short", "ascii_to_deva", "iast_full")
t_nohard  = run("abl_no_hardneg",   make_ds(train_pairs, use_hard=False))
t_noroman = run("abl_no_romanized", make_ds(train_pairs, drop=ROMAN))

tabs = {"base": base_tab, "ft_v1": ft_tab, "no_hardneg": t_nohard, "no_roman": t_noroman}
pd.DataFrame({k: v["MRR"] for k, v in tabs.items()}).round(3)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 0, 'pad_token_id': 1}.


Step,Training Loss
20,3.153025
40,2.911324
60,2.450102
80,2.019175
100,1.781731
120,1.680184
140,1.528037
160,1.442442
180,1.469221
200,1.347505


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 0, 'pad_token_id': 1}.


Step,Training Loss
20,3.645647
40,2.628546
60,1.858763
80,1.708723
100,1.594251
120,1.359628
140,1.259698
160,1.347854
180,1.301883
200,1.238102


,base,ft_v1,no_hardneg,no_roman
ascii_full,0.077,0.247,0.241,0.091
ascii_half,0.088,0.196,0.200,0.088
ascii_short,0.095,0.219,0.215,0.105
ascii_to_deva,0.376,0.862,0.869,0.164
deva_full,0.233,0.586,0.565,0.580
deva_half,0.227,0.446,0.446,0.443
english_to_deva,0.424,0.590,0.571,0.565
iast_full,0.089,0.253,0.242,0.082
q_casual,0.407,0.499,0.510,0.506
q_keyword,0.413,0.603,0.606,0.622


# Stage 4: test evaluation and failure analysis

## 4.1	Score base and ft_v1 on test, once, with confidence intervals

In [23]:
import numpy as np, pandas as pd
from sentence_transformers import SentenceTransformer

ft = SentenceTransformer(f"{ROOT}/models/e5_ft_v1/final", device="cuda")

b_out = evaluate(base, test_eval, test)
f_out = evaluate(ft,   test_eval, test)
b_out.to_csv(f"{RES}/baseline_test_ranks.csv", index=False)
f_out.to_csv(f"{RES}/ft_v1_test_ranks.csv", index=False)

b_tab, f_tab = summarize(b_out), summarize(f_out)
b_tab.to_csv(f"{RES}/baseline_test_metrics.csv"); f_tab.to_csv(f"{RES}/ft_v1_test_metrics.csv")

def boot_delta(b, f, n=2000, seed=0):
    rng = np.random.default_rng(seed)
    d = 1 / f["rank"].to_numpy() - 1 / b["rank"].to_numpy()
    g = pd.Series(d).groupby(b["gold_id"].to_numpy()).agg(["sum", "count"])
    s, c = g["sum"].to_numpy(), g["count"].to_numpy()
    m = [s[k].sum() / c[k].sum() for k in (rng.integers(0, len(g), len(g)) for _ in range(n))]
    lo, hi = np.percentile(m, [2.5, 97.5])
    return d.mean(), lo, hi

rows = {}
for name in sorted(b_out.qtype.unique()) + ["ALL"]:
    mask = (b_out.qtype == name) if name != "ALL" else np.ones(len(b_out), bool)
    d, lo, hi = boot_delta(b_out[mask], f_out[mask])
    rows[name] = {"n": int(mask.sum()),
                  "base R@1": b_tab.loc[name, "R@1"], "ft R@1": f_tab.loc[name, "R@1"],
                  "base MRR": b_tab.loc[name, "MRR"], "ft MRR": f_tab.loc[name, "MRR"],
                  "ΔMRR": d, "95% CI": f"[{lo:.3f}, {hi:.3f}]", "sig": lo > 0}
final = pd.DataFrame(rows).T
final.to_csv(f"{RES}/final_test_comparison.csv")
final

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

,n,base R@1,ft R@1,base MRR,ft MRR,ΔMRR,95% CI,sig
ascii_full,106,0.028,0.17,0.071,0.282,0.210326,"[0.145, 0.282]",True
ascii_half,98,0.051,0.082,0.095,0.161,0.065845,"[0.003, 0.126]",True
ascii_short,106,0.038,0.075,0.079,0.151,0.072331,"[0.023, 0.125]",True
ascii_to_deva,106,0.321,0.783,0.419,0.841,0.422491,"[0.346, 0.498]",True
deva_full,106,0.16,0.481,0.267,0.598,0.33027,"[0.241, 0.416]",True
deva_half,98,0.163,0.306,0.245,0.428,0.183798,"[0.115, 0.261]",True
english_to_deva,106,0.236,0.491,0.37,0.605,0.234728,"[0.152, 0.316]",True
iast_full,106,0.019,0.132,0.057,0.244,0.186152,"[0.127, 0.252]",True
q_casual,103,0.301,0.359,0.411,0.512,0.100519,"[0.039, 0.164]",True
q_keyword,106,0.274,0.462,0.422,0.591,0.169391,"[0.108, 0.236]",True


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

,n,base R@1,ft R@1,base MRR,ft MRR,ΔMRR,95% CI,sig
ascii_full,106,0.028,0.17,0.071,0.282,0.210323,"[0.145, 0.282]",True
ascii_half,98,0.051,0.082,0.095,0.161,0.065789,"[0.002, 0.126]",True
ascii_short,106,0.038,0.075,0.079,0.151,0.07233,"[0.023, 0.125]",True
ascii_to_deva,106,0.321,0.783,0.419,0.841,0.422491,"[0.346, 0.498]",True
deva_full,106,0.16,0.481,0.267,0.598,0.33027,"[0.241, 0.416]",True
deva_half,98,0.163,0.306,0.245,0.428,0.183895,"[0.115, 0.261]",True
english_to_deva,106,0.236,0.491,0.37,0.605,0.234728,"[0.152, 0.316]",True
iast_full,106,0.019,0.132,0.057,0.244,0.186145,"[0.127, 0.252]",True
q_casual,103,0.301,0.359,0.411,0.512,0.100519,"[0.039, 0.164]",True
q_keyword,106,0.274,0.462,0.422,0.591,0.169391,"[0.108, 0.236]",True


## Step 4.2: failure analysis

Questions this answers:

1. How often is the gold verse outside the top 10?
2. When the top-1 is wrong, is it an adjacent verse (a continuation of the same sentence)?
3. Which verses are hardest, and does verse length matter?
4. Which verse pairs get confused, and are they reasonable mistakes?

In [24]:
def predict(model, ev, corpus, qp="query: ", dp="passage: "):
    ids = corpus.id.tolist()
    text = {"english": corpus.english.tolist(), "deva": corpus.deva.tolist()}
    enc = lambda xs: model.encode(xs, normalize_embeddings=True, batch_size=64, show_progress_bar=False)
    docs = {s: enc([dp + t for t in text[s]]) for s in text}
    Q = enc([qp + q for q in ev["query"]])
    return [ids[int(np.argmax(docs[side] @ q))] for q, side in zip(Q, ev["doc_side"])]

f_out["pred"] = predict(ft, test_eval, test)      # same row order as test_eval
eng = dict(zip(test.id, test.english))

# 1. share of queries where the gold verse is outside the top 10
print("gold outside top-10:")
print(pd.DataFrame({"base": b_out.assign(f=b_out["rank"] > 10).groupby("qtype").f.mean(),
                    "ft":   f_out.assign(f=f_out["rank"] > 10).groupby("qtype").f.mean()}).round(2))

# 2. is the wrong top-1 an adjacent verse? (compared with chance)
ch = dict(zip(test.id, test.chapter)); vn = {i: int(i.split(".")[1]) for i in test.id}
adj = lambda a, b: ch[a] == ch[b] and abs(vn[a] - vn[b]) == 1
wrong = f_out[f_out["rank"] > 1]
obs = np.mean([adj(g, p) for g, p in zip(wrong.gold_id, wrong.pred)])
chance = np.mean([sum(adj(g, j) for j in test.id) / (len(test) - 1) for g in wrong.gold_id])
print(f"\nwrong top-1 is an adjacent verse: {obs:.3f} (chance {chance:.3f})")

# 3. hardest verses, and verse length vs performance
per = f_out.assign(rr=1 / f_out["rank"]).groupby("gold_id").rr.mean().to_frame("MRR")
per["eng_words"] = test.set_index("id").english.str.split().str.len()
print("\nSpearman(verse length, MRR):", round(per.corr(method="spearman").iloc[0, 1], 2))
print("\nhardest verses:")
for i, r in per.sort_values("MRR").head(8).iterrows():
    print(f"  {i}  MRR={r.MRR:.2f}  {eng[i][:110]}")

# 4. most frequent confusions
print("\nmost frequent confusions:")
for (g, p), n in wrong.groupby(["gold_id", "pred"]).size().sort_values(ascending=False).head(6).items():
    print(f"  {n}x  gold {g}: {eng[g][:90]}\n       got  {p}: {eng[p][:90]}")

gold outside top-10:
                 base    ft
qtype                      
ascii_full       0.85  0.52
ascii_half       0.85  0.67
ascii_short      0.87  0.69
ascii_to_deva    0.37  0.10
deva_full        0.47  0.15
deva_half        0.58  0.31
english_to_deva  0.38  0.17
iast_full        0.89  0.51
q_casual         0.37  0.19
q_keyword        0.23  0.15
q_natural        0.41  0.18

wrong top-1 is an adjacent verse: 0.095 (chance 0.018)

Spearman(verse length, MRR): -0.17

hardest verses:
  18.1  MRR=0.08  Arjuna asked the Almighty Krishna: Please explain to me, Dear Lord, what it means to have truly RENOUNCED (sur
  18.2  MRR=0.14  The Blessed Lord replied: If one entirely gives up all selfish motives, desires and actions, this is called tr
  18.9  MRR=0.16  But, O son of Kunti, he who partakes in these holy tasks, O Arjuna, only because he knows that it is his duty 
  18.12  MRR=0.17  When a man perform duties (or work) of any kind only for the sake of reward, the work at times eithe

In [25]:
# A. hubness: which passages are predicted wrongly most often?
print("most common WRONG predictions (ft):")
for i, n in wrong.pred.value_counts().head(8).items():
    print(f"  {i}: {n}x  {eng[i][:80]}")
print("expected per verse if uniform:", round(len(wrong) / len(test), 1))

# B. the X -> X0 confusions: which query types, and does the base model do it too?
b_out["pred"] = predict(base, test_eval, test)
pair = lambda df, g, p: df[(df.gold_id == g) & (df.pred == p)]
for g, p in [("18.4", "18.40"), ("18.6", "18.60"), ("18.7", "18.70")]:
    print(g, "->", p, "| ft:", pair(f_out, g, p).qtype.tolist())
    print("   base:", len(pair(b_out, g, p)), "queries")

# C. raw text of these verses: look for leftover digits or artifacts
for i in ["18.4", "18.40"]:
    r = test[test.id == i].iloc[0]
    print(i, "|", r.deva, "|", r.ascii)

most common WRONG predictions (ft):
  18.70: 31x  The Blessed Lord Krishna declared: He who studies and truly learns this sacred c
  18.20: 24x  When one can see Eternity, Infinity, and an undivided spiritual nature in things
  18.65: 23x  Devote your heart, mind, religious sacrifices and prayers to Me for eternity O P
  18.60: 22x  Although you do not want to accomplish your prescribed task, O Arjuna, you will 
  18.40: 18x  Nothing exists, either on the face of this earth or among the Demi- gods in heav
  18.68: 17x  However O Bharata, he who preaches My divine teachings to those who show true lo
  18.30: 17x  Pure and SAATVIC wisdom, O Partha, is that which one possesses when one knows wh
  17.17: 16x  If these three types of harmony are practised with supreme faith, a pure heart, 
expected per verse if uniform: 7.1
18.4 -> 18.40 | ft: ['deva_full', 'iast_full', 'ascii_full', 'deva_half', 'ascii_half', 'ascii_short', 'ascii_to_deva', 'english_to_deva', 'q_natural']
   base: 4 querie

In [26]:
full = pd.read_csv(f"{D}/gita_clean.csv", dtype={"id": str})
bad = full[full.deva.str.contains(r"\.\.\.|…") | full.english.str.contains(r"\.\.\.")]
print(len(bad), bad.id.tolist()[:20])

print("base most common wrong predictions:")
print(b_out[b_out["rank"] > 1].pred.value_counts().head(8))

148 ['1.1', '1.2', '1.20', '1.23', '1.26', '1.32', '1.35', '1.36', '2.3', '2.5', '2.6', '2.7', '2.8', '2.9', '2.11', '2.20', '2.22', '2.29', '2.31', '2.37']
base most common wrong predictions:
pred
18.29    180
18.68    102
18.67     67
18.60     33
17.9      32
17.20     31
18.40     26
18.1      24
Name: count, dtype: int64
148 ['1.1', '1.2', '1.20', '1.23', '1.26', '1.32', '1.35', '1.36', '2.3', '2.5', '2.6', '2.7', '2.8', '2.9', '2.11', '2.20', '2.22', '2.29', '2.31', '2.37']
base most common wrong predictions:
pred
18.29    180
18.68    102
18.67     67
18.60     33
17.9      32
17.20     31
18.40     26
18.1      24
Name: count, dtype: int64


## Step 4.3: qualitative before/after examples

In [27]:
txt = {"english": eng, "deva": dict(zip(test.id, test.deva))}

def show(qtype, kind, n=3, seed=0):
    m = (b_out.qtype == qtype)
    if kind == "fixed":  m &= (b_out["rank"] > 1) & (f_out["rank"] == 1)
    if kind == "failed": m &= (f_out["rank"] > 1)
    for i in b_out[m].sample(min(n, int(m.sum())), random_state=seed).index:
        side = b_out.doc_side[i]
        print(f"[{kind} | {qtype}] {b_out['query'][i][:90]}")
        print(f"  GOLD {b_out.gold_id[i]}: {txt[side][b_out.gold_id[i]][:90]}")
        print(f"  BASE rank {b_out['rank'][i]:>3} -> {b_out.pred[i]}: {txt[side][b_out.pred[i]][:70]}")
        print(f"  FT   rank {f_out['rank'][i]:>3} -> {f_out.pred[i]}: {txt[side][f_out.pred[i]][:70]}\n")

for qt in ["ascii_full", "deva_full", "q_natural"]:
    show(qt, "fixed")
for qt in ["ascii_full", "q_natural"]:
    show(qt, "failed")

[fixed | ascii_full] manah prasadah saumyatvam maunamatmavinigrahah bhavasamsuddhirityetattapo manasamucyate
  GOLD 17.16: Peace and tranquility of the mind, harmony and confidence in oneself, love, caring and gen
  BASE rank  37 -> 18.67: The Blessed Lord urged: Do not, O Partha, explain My teachings to thos
  FT   rank   1 -> 17.16: Peace and tranquility of the mind, harmony and confidence in oneself, 

[fixed | ascii_full] jnanam jneyam parijnata trividha karmacodana karanam karma karteti trividhah karmasamgraha
  GOLD 18.18: rjuna, there are three instigators (stimuli) of action in this world namely, knowledge its
  BASE rank  59 -> 18.29: Now Arjuna, hear and understand as I reveal to you the three divisions
  FT   rank   1 -> 18.18: rjuna, there are three instigators (stimuli) of action in this world n

[fixed | ascii_full] dhrtya yaya dharayate manahpranendriyakriyah yogenavyabhicarinya dhrtih sa partha sattviki
  GOLD 18.33: O Partha, while practising Yoga and full concentratio

## Step 4.4: out-of-domain check (Ayurveda pages)

In [28]:
import pandas as pd

In [29]:
!pip -q install indic-transliteration
import re, unicodedata
from indic_transliteration import sanscript
from indic_transliteration.sanscript import transliterate

S = [
 ("अथ आयुर्वेदः नाम शास्त्रम्", "Now, Ayurveda is the name of a science."),
 ("आयुर्वेदः नाम आयुः वेदयति इति आयुर्वेदः", "Ayurveda is so named because it makes one know life."),
 ("आयुः नाम शरीर इन्द्रिय सत्त्व आत्म संयोगः तस्य हिताहितं सुखं दुःखम् आयुः", "Life is the union of body, senses, mind and self; its benefit and harm, pleasure and pain, are also life."),
 ("तस्य हितं च अहितं च मानम् च तच् च यत्र उक्तं तत् आयुर्वेदः", "That in which the beneficial, the harmful and their measure are described is Ayurveda."),
 ("त्रयो दोषाः वातः पित्तं कफः इति", "There are three doshas: vata, pitta and kapha."),
 ("एते दोषाः शरीरस्य धारणम् कुर्वन्ति", "These doshas sustain the body."),
 ("वातः गतिः कारणम्", "Vata is the cause of movement."),
 ("पित्तं पाक कारणम्", "Pitta is the cause of digestion."),
 ("कफः स्थिरत्व कारणम्", "Kapha is the cause of stability."),
 ("आहारः महत्त्वं आयुः वर्धनम् करोति", "Diet is important; it promotes longevity."),
 ("सम्यक् आहारः शरीरं बलम् वर्धयति", "Proper diet increases the strength of the body."),
 ("असम्यक् आहारः रोग कारणम् भवति", "Improper diet becomes a cause of disease."),
 ("हिताहार सेवनं आयुः वर्धयति", "Eating wholesome food increases lifespan."),
 ("दिनचर्या नाम नित्य कर्म अनुष्ठानम्", "Daily routine means performing regular daily duties."),
 ("प्रातः उत्थानं शौचं दन्त धावनं कर्तव्यम्", "In the morning one should wake up, attend to cleansing and brush the teeth."),
 ("व्यायामः शरीरस्य बल वर्धनं करोति", "Exercise increases the strength of the body."),
 ("स्नानं शरीर शुद्धि कारणम्", "Bathing is the cause of purification of the body."),
 ("रोगः नाम दोष वैषम्य कारणम्", "Disease is caused by imbalance of the doshas."),
 ("दोष सम्यता आरोग्य कारणम्", "Balance of the doshas is the cause of health."),
 ("रोग उत्पत्ति हेतवः मिथ्या आहार विहारः", "The causes of the origin of disease are wrong diet and lifestyle."),
 ("रोग निवारणम् सम्यक् चिकित्सा द्वारा भवति", "Disease is removed through proper treatment."),
]

def to_ascii(s):
    s = unicodedata.normalize("NFD", s)
    s = "".join(c for c in s if not unicodedata.combining(c))
    s = s.replace("'", "").replace("-", "").replace("|", " ")
    return re.sub(r"\s+", " ", s).strip()

corpus = pd.DataFrame({"id": [str(i + 1) for i in range(len(S))],
                       "deva": [a for a, _ in S], "english": [b for _, b in S]})

rows = []
for i, (sa, _) in enumerate(S):
    g = str(i + 1)
    rows.append(dict(gold_id=g, qtype="deva_full", query=sa, doc_side="english"))
    rows.append(dict(gold_id=g, qtype="ascii_full", doc_side="english",
                     query=to_ascii(transliterate(sa, sanscript.DEVANAGARI, sanscript.IAST))))
QS = [("What are the three doshas?", 5), ("Which dosha is responsible for digestion?", 8),
      ("What is the meaning of the word Ayurveda?", 2), ("How does proper diet help the body?", 11),
      ("What should a person do in the morning?", 15), ("Why is exercise good for you?", 16),
      ("What lifestyle habits lead to illness?", 20), ("What gives stability to the body?", 9)]
rows += [dict(gold_id=str(g), qtype="q_english", query=q, doc_side="english") for q, g in QS]
ev_ood = pd.DataFrame(rows)

b = summarize(evaluate(base, ev_ood, corpus))
f = summarize(evaluate(ft,   ev_ood, corpus))
pd.concat({"base": b[["n", "R@1", "R@5", "MRR"]], "ft": f[["R@1", "R@5", "MRR"]]}, axis=1)

base                          ft              
               n    R@1    R@5    MRR    R@1    R@5    MRR
ascii_full  21.0  0.286  0.476  0.402  0.286  0.619  0.457
deva_full   21.0  0.857  1.000  0.929  0.857  1.000  0.921
q_english    8.0  0.750  1.000  0.854  0.750  1.000  0.854
ALL         50.0  0.600  0.780  0.696  0.600  0.840  0.715

base                          ft              
               n    R@1    R@5    MRR    R@1    R@5    MRR
ascii_full  21.0  0.286  0.476  0.402  0.286  0.619  0.457
deva_full   21.0  0.857  1.000  0.929  0.857  1.000  0.921
q_english    8.0  0.750  1.000  0.854  0.750  1.000  0.854
ALL         50.0  0.600  0.780  0.696  0.600  0.840  0.715

In [30]:
import os
P = "/content/drive/MyDrive/sanskrit_retrieval/models/e5_ft_v1/final"
print(os.listdir(P))
print(round(sum(os.path.getsize(os.path.join(P, f)) for f in os.listdir(P)) / 1e6), "MB")

['config_sentence_transformers.json', 'config.json', 'model.safetensors', 'tokenizer_config.json', 'tokenizer.json', 'sentence_bert_config.json', '1_Pooling', '2_Normalize', 'modules.json', 'README.md']
488 MB
